In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from datetime import datetime, timedelta

import pandas as pd
import requests
import json
import time

In [3]:
#pip show selenium

In [8]:
#pip show pandas

In [9]:
#pip show requests

In [2]:
# ==========================================
# CONFIGURACIÓN
# ==========================================

# URL del metal 
metal_url = "https://www.lme.com/en/metals/non-ferrous/lme-aluminium#Price+graphs"

# datasource del gráfico Official Prices
official_datasource_id = "dddbc815-1a81-4f35-beed-6a193f4c946a"

# datasource del gráfico Closing Prices
closing_datasource_id = "37cf78d1-222d-4618-b9f4-c5f13ff421e6"

# Periodo en años de extracción de datos
years = 5

# token de consulta de banxico
BANXICO_TOKEN = "8597637d49292ac0004ebf2efdc0bfec10d2a1794f28f3c60c7be2ff65446961"

#serie para consulta del tipo de cambio USD-MXN (fix)
SERIE = "SF43718"



In [3]:
#funcion para iniciar navegador chrome usando Selenium
def start_driver():
    options = Options()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    return driver

In [4]:
#función para extraer la serie histórica de la gráfica de los precios oficiales
#utiliza el navegador creado con la función start_driver 
def download_official_history(driver, datasource_id, years=5):
    
    end_date = datetime.today()
    start_date = end_date - timedelta(days=years*365)
    
    all_rows = []
    current_start = start_date

    #ciclo para dividir el periodo total en periodos más pequeños y evitar bloqueo del intento de scraping
    while current_start < end_date:
        
        current_end = current_start + timedelta(days=180)
        if current_end > end_date:
            current_end = end_date
        
        print(f"Descargando desde {current_start.date()} hasta {current_end.date()}")

        #script en JavaScript para usar la API del LME y el ID del metal para extraer la información de la gráfica.  
        #guarda la infomración extaida en la variable extraida como texto plano
        script = f"""
        var callback = arguments[arguments.length - 1];
        fetch("https://www.lme.com/api/trading-data/chart-data?datasourceId={datasource_id}&startDate={current_start.strftime('%Y-%m-%d')}&endDate={current_end.strftime('%Y-%m-%d')}")
          .then(response => response.text())
          .then(data => callback(data))
          .catch(error => callback("ERROR: " + error));
        """
        #correr el script anterior en el navegador creado con Selenium
        #se usa execute_async_script para asegurar el código continue hasta tener una respuesta de la API del LME
        result = driver.execute_async_script(script)

        #en caso de error: 
        if not result or result.startswith("ERROR") or result.strip().startswith("<"):
            print("Respuesta inválida de la API")
            print(str(result)[:300])
            break

        #convertir texto plano en objeto de de python 
        data = json.loads(result)

        
        labels = data["Labels"]
        datasets = data["Datasets"]

        #estructurar la información iterando por cada dia  
        for i, date_str in enumerate(labels):

            #crear fila convirtiendo la fecha de texto plano a fecha
            row = {
                "date": datetime.strptime(date_str, "%d/%m/%Y")
            }

            #recorrer por cada dataset, cada dataset es un tipo de precio/contrato
            for dataset in datasets:

                #estructuración del título de la columna. Formato:  tipodecontrato_tipodeprecio
                contract = dataset["RowTitle"].lower().replace(" ", "-")
                price_type = dataset["Label"].lower()
                
                column_name = f"{contract}_{price_type}"
                
                row[column_name] = dataset["Data"][i]
            
            all_rows.append(row)
        
        current_start = current_end + timedelta(days=1)
    
    #guardar infomración en un DF
    df = pd.DataFrame(all_rows)

    #limpieza de duplicados y ordenar por fecha
    df = df.sort_values("date").drop_duplicates(subset="date")
    
    return df

In [5]:
#función para extraer la serie histórica de la gráfica de los precios de cierre. 
#misms estructura que la función download_official_history
def download_closing_history(driver, datasource_id, years=5):
    
    end_date = datetime.today()
    start_date = end_date - timedelta(days=years*365)
    
    all_rows = []
    current_start = start_date
    
    while current_start < end_date:
        
        current_end = current_start + timedelta(days=180)
        if current_end > end_date:
            current_end = end_date
        
        print(f"Descargando {current_start.date()} → {current_end.date()}")
        
        script = f"""
        var callback = arguments[arguments.length - 1];
        fetch("https://www.lme.com/api/trading-data/chart-data?datasourceId={datasource_id}&startDate={current_start.strftime('%Y-%m-%d')}&endDate={current_end.strftime('%Y-%m-%d')}")
          .then(response => response.text())
          .then(data => callback(data))
          .catch(error => callback("ERROR: " + error));
        """
        
        result = driver.execute_async_script(script)
        
        if not result or result.startswith("ERROR") or result.strip().startswith("<"):
            print("Respuesta inválida de la API")
            print(str(result)[:300])
            break
        
        data = json.loads(result)
        
        labels = data["Labels"]
        datasets = data["Datasets"]
        
        for i, date_str in enumerate(labels):
            
            row = {
                "date": datetime.strptime(date_str, "%d/%m/%Y")
            }
            
            for dataset in datasets:
                
                contract = dataset["RowTitle"].lower().replace(" ", "-")
                price_type = dataset["Label"].lower()
                
                column_name = f"{contract}_{price_type}"
                
                row[column_name] = dataset["Data"][i]
            
            all_rows.append(row)
        
        current_start = current_end + timedelta(days=1)
    
    df = pd.DataFrame(all_rows)
    
    df = df.sort_values("date").drop_duplicates(subset="date")
    
    return df

In [6]:
#extracción de información de precios oficiales
driver = start_driver()

driver.get(metal_url)

#tiempo de espera para asegurar haya cargado el buscador
time.sleep(5)

df_official = download_official_history(driver, official_datasource_id, years)

driver.quit()

df_official.tail()

Descargando 2021-03-15 → 2021-09-11
Descargando 2021-09-12 → 2022-03-11
Descargando 2022-03-12 → 2022-09-08
Descargando 2022-09-09 → 2023-03-08
Descargando 2023-03-09 → 2023-09-05
Descargando 2023-09-06 → 2024-03-04
Descargando 2024-03-05 → 2024-09-01
Descargando 2024-09-02 → 2025-03-01
Descargando 2025-03-02 → 2025-08-29
Descargando 2025-08-30 → 2026-02-26
Descargando 2026-02-27 → 2026-03-14


,date,cash_bid,cash_offer,3-month_bid,3-month_offer,dec-27_bid,dec-27_offer,dec-28_bid,dec-28_offer,dec-29_bid,dec-29_offer
1262,2026-03-09,3406.00,3406.50,3385.00,3387.00,3177.00,3182.00,3077.00,3082.00,2982.00,2987.00
1263,2026-03-10,3402.00,3402.50,3388.00,3390.00,3165.00,3170.00,3070.00,3075.00,2955.00,2960.00
1264,2026-03-11,3465.00,3467.00,3442.00,3444.00,3153.00,3158.00,3045.00,3050.00,2940.00,2945.00
1265,2026-03-12,3516.00,3516.50,3492.00,3494.00,3118.00,3123.00,2963.00,2968.00,2848.00,2853.00
1266,2026-03-13,3519.50,3520.00,3485.50,3486.00,3017.00,3022.00,2832.00,2837.00,2697.00,2702.00


In [8]:
driver = start_driver()

driver.get(metal_url)

time.sleep(5)

df_closing = download_closing_history(driver, closing_datasource_id, years)

driver.quit()

df_closing.tail()

Descargando 2021-03-15 → 2021-09-11
Descargando 2021-09-12 → 2022-03-11
Descargando 2022-03-12 → 2022-09-08
Descargando 2022-09-09 → 2023-03-08
Descargando 2023-03-09 → 2023-09-05
Descargando 2023-09-06 → 2024-03-04
Descargando 2024-03-05 → 2024-09-01
Descargando 2024-09-02 → 2025-03-01
Descargando 2025-03-02 → 2025-08-29
Descargando 2025-08-30 → 2026-02-26
Descargando 2026-02-27 → 2026-03-14


,date,3-month_price,month-1_price,month-2_price,month-3_price,month-4_price,month-5_price,month-6_price
1262,2026-03-09,3385.50,3410.04,3405.79,3391.30,3371.37,3354.30,3346.30
1263,2026-03-10,3406.00,3431.78,3433.74,3417.40,3399.19,3385.94,3382.44
1264,2026-03-11,3457.00,3484.36,3487.33,3469.16,3451.16,3433.41,3425.16
1265,2026-03-12,3516.50,3549.39,3546.15,3530.97,3511.03,3491.53,3477.78
1266,2026-03-13,3439.50,3470.41,3465.34,3451.58,3432.23,3408.23,3389.73


In [9]:
df_closing.tail(25)

,date,3-month_price,month-1_price,month-2_price,month-3_price,month-4_price,month-5_price,month-6_price
1242,2026-02-09,3125.50,3105.51,3116.52,3122.56,3126.13,3129.50,3131.75
1243,2026-02-10,3093.00,3064.61,3081.98,3089.14,3093.72,3097.22,3099.47
1244,2026-02-11,3103.00,3075.24,3092.58,3099.70,3103.94,3107.08,3107.00
1245,2026-02-12,3100.00,3068.58,3087.75,3096.26,3101.77,3104.77,3107.27
1246,2026-02-13,3077.50,3043.01,3064.49,3073.38,3079.72,3083.22,3085.22
1247,2026-02-16,3052.50,3039.02,3047.71,3054.11,3057.54,3058.29,3060.54
1248,2026-02-17,3035.00,3020.32,3029.28,3035.39,3039.39,3040.89,3043.14
1249,2026-02-18,3089.00,3074.27,3083.22,3088.48,3092.15,3093.40,3095.40
1250,2026-02-19,3067.50,3055.70,3062.57,3067.53,3070.37,3070.62,3071.12
1251,2026-02-20,3102.50,3092.38,3097.75,3102.50,3105.65,3106.65,3106.00


In [10]:
#extracción del tipo de cambio usando API de Banxico
def download_fx_history(years=5):
    
    end_date = datetime.today()
    start_date = end_date - timedelta(days=years*365)

    #URL para llamar API
    url = (
        "https://www.banxico.org.mx/SieAPIRest/service/v1/series/"
        f"{SERIE}/datos/"
        f"{start_date.strftime('%Y-%m-%d')}/{end_date.strftime('%Y-%m-%d')}"
    )
    
    headers = {
        "Bmx-Token": BANXICO_TOKEN
    }

    #petición usando requests
    response = requests.get(url, headers=headers)

    #convertir a un objeto python
    data = response.json()
    
    series = data["bmx"]["series"][0]["datos"]
    
    rows = []
    #ciclo para generar las filas iterando por cada entrada
    for entry in series:
        
        #ignorar datos inválidos
        if entry["dato"] != "N/E":
            
            rows.append({
                "date": datetime.strptime(entry["fecha"], "%d/%m/%Y"),
                "usd_mxn_fix": float(entry["dato"])
            })
    
    df = pd.DataFrame(rows)
    
    df = df.sort_values("date")
    
    return df

In [11]:
df_fx = download_fx_history(years)

df_fx.tail(25)

,date,usd_mxn_fix
1234,2026-02-09,17.1907
1235,2026-02-10,17.2315
1236,2026-02-11,17.2155
1237,2026-02-12,17.2000
1238,2026-02-13,17.1798
1239,2026-02-16,17.1723
1240,2026-02-17,17.1753
1241,2026-02-18,17.1392
1242,2026-02-19,17.2700
1243,2026-02-20,17.1722


In [12]:
#asegurar todas las columnas de fechas estpen bajo el mismo formato
df_official["date"] = pd.to_datetime(df_official["date"])
df_closing["date"] = pd.to_datetime(df_closing["date"])
df_fx["date"] = pd.to_datetime(df_fx["date"])

In [13]:
#unir los 3 df en uno solo usando la fecha como clave
df_global = (
    df_official
    .merge(df_closing, on="date", how="outer")
    .merge(df_fx, on="date", how="outer")
)

In [14]:
#ordenar por fecha 
df_global = df_global.sort_values("date")

In [16]:
df_global.tail(40)

,date,cash_bid,cash_offer,3-month_bid,3-month_offer,dec-27_bid,dec-27_offer,dec-28_bid,dec-28_offer,dec-29_bid,dec-29_offer,3-month_price,month-1_price,month-2_price,month-3_price,month-4_price,month-5_price,month-6_price,usd_mxn_fix
1253,2026-01-19,3168.00,3168.50,3150.00,3152.00,3138.00,3143.00,3118.00,3123.00,3098.00,3103.00,3158.50,3148.55,3155.22,3160.80,3164.23,3166.23,3168.23,17.5990
1254,2026-01-20,3136.50,3137.00,3135.50,3136.00,3102.00,3107.00,3068.00,3073.00,3048.00,3053.00,3107.50,3092.42,3101.09,3107.88,3113.99,3116.49,3118.49,17.6008
1255,2026-01-21,3112.00,3114.00,3122.50,3123.00,3110.00,3115.00,3085.00,3090.00,3065.00,3070.00,3115.00,3098.71,3106.92,3113.93,3120.23,3122.92,3123.92,17.4520
1256,2026-01-22,3093.00,3093.50,3113.50,3114.00,3097.00,3102.00,3068.00,3073.00,3047.00,3052.00,3132.50,3116.98,3125.04,3132.02,3137.85,3140.35,3142.35,17.4828
1257,2026-01-23,3174.50,3175.00,3176.00,3177.00,3147.00,3152.00,3113.00,3118.00,3092.00,3097.00,3169.00,3161.14,3166.74,3170.36,3172.27,3172.77,3170.77,17.4545
1258,2026-01-26,3190.00,3192.00,3195.00,3195.50,3133.00,3138.00,3093.00,3098.00,3060.00,3065.00,3188.50,3177.82,3182.99,3188.06,3191.54,3192.00,3190.00,17.2830
1259,2026-01-27,3166.00,3166.50,3177.50,3178.00,3105.00,3110.00,3065.00,3070.00,3038.00,3043.00,3207.00,3198.46,3202.26,3206.27,3209.72,3210.22,3209.00,17.2357
1260,2026-01-28,3258.00,3259.00,3261.00,3262.00,3138.00,3143.00,3058.00,3063.00,2983.00,2988.00,3257.00,3247.94,3251.45,3255.29,3259.25,3259.75,3256.75,17.2322
1261,2026-01-29,3323.00,3325.00,3325.50,3326.00,3173.00,3178.00,3048.00,3053.00,2943.00,2948.00,3218.50,3202.56,3210.72,3216.95,3219.18,3221.18,3217.43,17.2532
1262,2026-01-30,3109.00,3110.00,3132.00,3134.00,3068.00,3073.00,3003.00,3008.00,2938.00,2943.00,3144.00,3129.01,3137.91,3143.68,3146.05,3148.55,3145.05,17.3310


In [17]:
#completar fechas faltantes
full_dates = pd.date_range(
    start=df_global["date"].min(),
    end=df_global["date"].max(),
    freq="D"
)
df_global = df_global.set_index("date").reindex(full_dates)
df_global.index.name = "date"

In [18]:
df_global.tail(45)

,cash_bid,cash_offer,3-month_bid,3-month_offer,dec-27_bid,dec-27_offer,dec-28_bid,dec-28_offer,dec-29_bid,dec-29_offer,3-month_price,month-1_price,month-2_price,month-3_price,month-4_price,month-5_price,month-6_price,usd_mxn_fix
date,,,,,,,,,,,,,,,,,,
2026-01-28,3258.00,3259.00,3261.00,3262.00,3138.00,3143.00,3058.00,3063.00,2983.00,2988.00,3257.00,3247.94,3251.45,3255.29,3259.25,3259.75,3256.75,17.2322
2026-01-29,3323.00,3325.00,3325.50,3326.00,3173.00,3178.00,3048.00,3053.00,2943.00,2948.00,3218.50,3202.56,3210.72,3216.95,3219.18,3221.18,3217.43,17.2532
2026-01-30,3109.00,3110.00,3132.00,3134.00,3068.00,3073.00,3003.00,3008.00,2938.00,2943.00,3144.00,3129.01,3137.91,3143.68,3146.05,3148.55,3145.05,17.3310
2026-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-02-02,3041.00,3042.00,3065.00,3066.00,3003.00,3008.00,2953.00,2958.00,2903.00,2908.00,3056.00,3045.52,3050.78,3055.70,3057.61,3059.61,3058.11,NaN
2026-02-03,3102.00,3103.00,3112.50,3113.00,3035.00,3040.00,2965.00,2970.00,2913.00,2918.00,3106.50,3095.13,3098.51,3104.37,3108.01,3110.51,3110.50,17.2350
2026-02-04,3047.00,3047.50,3075.00,3076.00,3025.00,3030.00,2968.00,2973.00,2903.00,2908.00,3069.50,3052.85,3059.88,3066.60,3072.06,3075.06,3077.06,17.2925
2026-02-05,3013.50,3014.00,3036.00,3037.00,3003.00,3008.00,2960.00,2965.00,2920.00,2925.00,3027.00,3009.18,3016.78,3023.37,3029.42,3032.67,3033.17,17.4070


In [19]:
#Crear columnas de datos boleanos para identifar valores que fueron nulos originalmente
#identificar valores nulos para datos extraidos del LME
df_global["interpolated_LME"] = df_global["cash_bid"].isna()
#identicar valores nulos para datos extraidos de tipo de cambio
df_global["interpolated_MXN"] = df_global["usd_mxn_fix"].isna()

In [20]:
df_global.tail(45)

,cash_bid,cash_offer,3-month_bid,3-month_offer,dec-27_bid,dec-27_offer,dec-28_bid,dec-28_offer,dec-29_bid,dec-29_offer,3-month_price,month-1_price,month-2_price,month-3_price,month-4_price,month-5_price,month-6_price,usd_mxn_fix,interpolated_LME,interpolated_MXN
date,,,,,,,,,,,,,,,,,,,,
2026-01-28,3258.00,3259.00,3261.00,3262.00,3138.00,3143.00,3058.00,3063.00,2983.00,2988.00,3257.00,3247.94,3251.45,3255.29,3259.25,3259.75,3256.75,17.2322,False,False
2026-01-29,3323.00,3325.00,3325.50,3326.00,3173.00,3178.00,3048.00,3053.00,2943.00,2948.00,3218.50,3202.56,3210.72,3216.95,3219.18,3221.18,3217.43,17.2532,False,False
2026-01-30,3109.00,3110.00,3132.00,3134.00,3068.00,3073.00,3003.00,3008.00,2938.00,2943.00,3144.00,3129.01,3137.91,3143.68,3146.05,3148.55,3145.05,17.3310,False,False
2026-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True
2026-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,True
2026-02-02,3041.00,3042.00,3065.00,3066.00,3003.00,3008.00,2953.00,2958.00,2903.00,2908.00,3056.00,3045.52,3050.78,3055.70,3057.61,3059.61,3058.11,NaN,False,True
2026-02-03,3102.00,3103.00,3112.50,3113.00,3035.00,3040.00,2965.00,2970.00,2913.00,2918.00,3106.50,3095.13,3098.51,3104.37,3108.01,3110.51,3110.50,17.2350,False,False
2026-02-04,3047.00,3047.50,3075.00,3076.00,3025.00,3030.00,2968.00,2973.00,2903.00,2908.00,3069.50,3052.85,3059.88,3066.60,3072.06,3075.06,3077.06,17.2925,False,False
2026-02-05,3013.50,3014.00,3036.00,3037.00,3003.00,3008.00,2960.00,2965.00,2920.00,2925.00,3027.00,3009.18,3016.78,3023.37,3029.42,3032.67,3033.17,17.4070,False,False


In [21]:
#imputación de datos faltantes (se completan con el dato inmediato anterior) 
df_global = df_global.ffill()

In [22]:
df_global = df_global.reset_index()

In [23]:
df_global.tail(45)

,date,cash_bid,cash_offer,3-month_bid,3-month_offer,dec-27_bid,dec-27_offer,dec-28_bid,dec-28_offer,dec-29_bid,...,3-month_price,month-1_price,month-2_price,month-3_price,month-4_price,month-5_price,month-6_price,usd_mxn_fix,interpolated_LME,interpolated_MXN
1780,2026-01-28,3258.00,3259.00,3261.00,3262.00,3138.00,3143.00,3058.00,3063.00,2983.00,...,3257.00,3247.94,3251.45,3255.29,3259.25,3259.75,3256.75,17.2322,False,False
1781,2026-01-29,3323.00,3325.00,3325.50,3326.00,3173.00,3178.00,3048.00,3053.00,2943.00,...,3218.50,3202.56,3210.72,3216.95,3219.18,3221.18,3217.43,17.2532,False,False
1782,2026-01-30,3109.00,3110.00,3132.00,3134.00,3068.00,3073.00,3003.00,3008.00,2938.00,...,3144.00,3129.01,3137.91,3143.68,3146.05,3148.55,3145.05,17.3310,False,False
1783,2026-01-31,3109.00,3110.00,3132.00,3134.00,3068.00,3073.00,3003.00,3008.00,2938.00,...,3144.00,3129.01,3137.91,3143.68,3146.05,3148.55,3145.05,17.3310,True,True
1784,2026-02-01,3109.00,3110.00,3132.00,3134.00,3068.00,3073.00,3003.00,3008.00,2938.00,...,3144.00,3129.01,3137.91,3143.68,3146.05,3148.55,3145.05,17.3310,True,True
1785,2026-02-02,3041.00,3042.00,3065.00,3066.00,3003.00,3008.00,2953.00,2958.00,2903.00,...,3056.00,3045.52,3050.78,3055.70,3057.61,3059.61,3058.11,17.3310,False,True
1786,2026-02-03,3102.00,3103.00,3112.50,3113.00,3035.00,3040.00,2965.00,2970.00,2913.00,...,3106.50,3095.13,3098.51,3104.37,3108.01,3110.51,3110.50,17.2350,False,False
1787,2026-02-04,3047.00,3047.50,3075.00,3076.00,3025.00,3030.00,2968.00,2973.00,2903.00,...,3069.50,3052.85,3059.88,3066.60,3072.06,3075.06,3077.06,17.2925,False,False
1788,2026-02-05,3013.50,3014.00,3036.00,3037.00,3003.00,3008.00,2960.00,2965.00,2920.00,...,3027.00,3009.18,3016.78,3023.37,3029.42,3032.67,3033.17,17.4070,False,False
1789,2026-02-06,3044.50,3045.00,3063.00,3063.50,3040.00,3045.00,3005.00,3010.00,2970.00,...,3085.00,3069.65,3076.49,3081.61,3085.78,3088.49,3089.00,17.2988,False,False


In [24]:
#Aseguar todos los datos sean numéricos excepto los tipo boolenao y tipo fecha
cols_to_convert = df_global.columns.difference(
    ["date", "interpolated_LME", "interpolated_MXN"]
)
df_global[cols_to_convert] = df_global[cols_to_convert].apply(
    pd.to_numeric, errors="coerce"
)

In [25]:
#Cáculo del precio de contado en efectivo en peso mexicano
df_global["cash_offer_mxn"] = (
    df_global["cash_offer"] * df_global["usd_mxn_fix"]
).round(4)

In [26]:
#reordenar df
cols = df_global.columns.tolist()
cols.remove("usd_mxn_fix")
date_index = cols.index("date")
cols.insert(date_index + 1, "usd_mxn_fix")
df_global = df_global[cols]

In [27]:
df_global.tail()

,date,usd_mxn_fix,cash_bid,cash_offer,3-month_bid,3-month_offer,dec-27_bid,dec-27_offer,dec-28_bid,dec-28_offer,...,3-month_price,month-1_price,month-2_price,month-3_price,month-4_price,month-5_price,month-6_price,interpolated_LME,interpolated_MXN,cash_offer_mxn
1820,2026-03-09,17.7687,3406.0,3406.5,3385.0,3387.0,3177.0,3182.0,3077.0,3082.0,...,3385.5,3410.04,3405.79,3391.30,3371.37,3354.30,3346.30,False,False,60529.0766
1821,2026-03-10,17.5037,3402.0,3402.5,3388.0,3390.0,3165.0,3170.0,3070.0,3075.0,...,3406.0,3431.78,3433.74,3417.40,3399.19,3385.94,3382.44,False,False,59556.3392
1822,2026-03-11,17.6543,3465.0,3467.0,3442.0,3444.0,3153.0,3158.0,3045.0,3050.0,...,3457.0,3484.36,3487.33,3469.16,3451.16,3433.41,3425.16,False,False,61207.4581
1823,2026-03-12,17.8368,3516.0,3516.5,3492.0,3494.0,3118.0,3123.0,2963.0,2968.0,...,3516.5,3549.39,3546.15,3530.97,3511.03,3491.53,3477.78,False,False,62723.1072
1824,2026-03-13,17.9218,3519.5,3520.0,3485.5,3486.0,3017.0,3022.0,2832.0,2837.0,...,3439.5,3470.41,3465.34,3451.58,3432.23,3408.23,3389.73,False,False,63084.7360


In [28]:
df_global.info()

<class 'pandas.DataFrame'>
RangeIndex: 1825 entries, 0 to 1824
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              1825 non-null   datetime64[us]
 1   usd_mxn_fix       1824 non-null   float64       
 2   cash_bid          1825 non-null   float64       
 3   cash_offer        1825 non-null   float64       
 4   3-month_bid       1825 non-null   float64       
 5   3-month_offer     1825 non-null   float64       
 6   dec-27_bid        1825 non-null   float64       
 7   dec-27_offer      1825 non-null   float64       
 8   dec-28_bid        1825 non-null   float64       
 9   dec-28_offer      1825 non-null   float64       
 10  dec-29_bid        1825 non-null   float64       
 11  dec-29_offer      1825 non-null   float64       
 12  3-month_price     1825 non-null   float64       
 13  month-1_price     1825 non-null   float64       
 14  month-2_price     1825 non-null   f

In [32]:
#guardar infomración en un archivo CSV

start_date_str = df_global["date"].min().strftime("%Y-%m-%d")
today_str = datetime.today().strftime("%Y-%m-%d")
#formato de título "global_file_fechainicial_fechafinal.csv"
global_file = f"global_file_{start_date_str}_to_{today_str}.csv"

df_global.to_csv(global_file, index=False)

print("Archivo guardado:", global_file)

Archivo guardado: global_file_2021-03-15_to_2026-03-14.csv
